# PNGF on Google Colab
**Precomputed Numerical Green Function — EM Inverse Design**

## Before running anything
1. Go to **Runtime → Change runtime type**
2. Set **Runtime shape** to `High-RAM` → **Save**
3. The `acmefdfd` step needs ~50 GB RAM

## Workflow
| Cell | Step | Notes |
|------|------|-------|
| §0 | RAM check | Must show ≥40 GB |
| §1 | Install deps | ~3 min |
| §2 | Mount Drive | Optional, persists Gmat files |
| §3 | Clone repo | From GitHub |
| §4 | Build acmefdfd | ~1 min |
| §5 | Build pngf-opt | ~30 sec |
| §6 | SSH tunnel | Cursor Remote-SSH access |
| §7 | Run acmefdfd | **~1–4 hrs, 50 GB RAM** |
| §8 | Run pngf-opt | Minutes |

In [ ]:
# §0  Check available RAM
import subprocess
result = subprocess.run(['cat', '/proc/meminfo'], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if line.startswith('MemTotal'):
        gb = int(line.split()[1]) / 1e6
        symbol = '✅' if gb >= 40 else '❌'
        print(f"{symbol}  Total RAM: {gb:.1f} GB")
        if gb < 40:
            print("   Go to Runtime → Change runtime type → High-RAM and reconnect.")
        break

In [ ]:
# §1  Install system dependencies
!apt-get update -qq
!apt-get install -y -qq \
    gcc g++ gfortran \
    libopenmpi-dev openmpi-bin \
    libeigen3-dev \
    libmetis-dev \
    libscalapack-openmpi-dev \
    libmumps-dev \
    libopenblas-dev \
    liblapack-dev
print('✅ All dependencies installed.')

In [ ]:
# §2  Mount Google Drive (recommended — persists Gmat files across sessions)
# Skip this cell if you do not want to use Drive.
from google.colab import drive
import os
drive.mount('/content/drive')
GDRIVE_PNGF = '/content/drive/MyDrive/PNGF_outputs'
os.makedirs(GDRIVE_PNGF, exist_ok=True)
print(f'✅ Drive mounted. Gmat files will be saved to: {GDRIVE_PNGF}')

In [ ]:
# §3  Clone repository
import os
REPO_DIR = '/root/PNGF'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/KesavaViswanadha/PNGF.git {REPO_DIR}
else:
    print('Repo already present — pulling latest.')
    !git -C {REPO_DIR} pull
print('✅ Repo ready at', REPO_DIR)
!ls {REPO_DIR}

In [ ]:
# §4  Build acmefdfd
import os
%cd /root/PNGF/acmefdfd
!make clean && make
if os.path.isfile('acmefdfd'):
    print('\n✅ acmefdfd built successfully.')
else:
    print('\n❌ Build failed — check output above.')

In [ ]:
# §5  Build pngf-opt
import os
%cd /root/PNGF/pngf-opt
!make clean && make
if os.path.isfile('pngf-opt'):
    print('\n✅ pngf-opt built successfully.')
else:
    print('\n❌ Build failed — check output above.')

In [ ]:
# §6  SSH tunnel — connect Cursor to this Colab VM
#
# Requires a FREE ngrok account:
#   1. Sign up at https://ngrok.com
#   2. Copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
#   3. Paste it in NGROK_TOKEN below

NGROK_TOKEN = ''   # paste your token here
SSH_PASSWORD = 'pngf-colab'   # change if you like

if not NGROK_TOKEN:
    print('Set NGROK_TOKEN above first, then re-run this cell.')
else:
    import os
    !apt-get install -y -qq openssh-server > /dev/null 2>&1
    !mkdir -p /run/sshd
    os.system(f'echo "root:{SSH_PASSWORD}" | chpasswd')
    !sed -i 's/#PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
    !grep -q 'PasswordAuthentication yes' /etc/ssh/sshd_config || echo 'PasswordAuthentication yes' >> /etc/ssh/sshd_config
    !/usr/sbin/sshd
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    tunnel = ngrok.connect(22, 'tcp')
    addr = tunnel.public_url.replace('tcp://', '')
    host, port = addr.split(':')
    print(f"""
====================================================
SSH tunnel active!

Add to ~/.ssh/config on your Mac:

  Host colab-pngf
    HostName {host}
    User root
    Port {port}

In Cursor: Cmd+Shift+P → Remote-SSH: Connect to Host → colab-pngf
Password: {SSH_PASSWORD}
Open folder: /root/PNGF
====================================================
""")

In [ ]:
# §7  Run acmefdfd — generate G matrices
# Requires High-RAM runtime (~50 GB peak).  Runtime: ~1-4 hours.
import os, shutil
%cd /root/PNGF/acmefdfd
!chmod +x run_acmefdfd.sh
!./run_acmefdfd.sh
GDRIVE_PNGF = '/content/drive/MyDrive/PNGF_outputs'
if os.path.isdir(GDRIVE_PNGF):
    gmat_files = [f for f in os.listdir('../pngf-opt/') if f.startswith('Gmat')]
    for f in gmat_files:
        shutil.copy(f'../pngf-opt/{f}', f'{GDRIVE_PNGF}/{f}')
    print(f'✅ Copied {len(gmat_files)} Gmat file(s) to Google Drive.')
print('\n✅ acmefdfd complete.')

In [ ]:
# §8  Run pngf-opt — inverse design optimization
import os, shutil
GDRIVE_PNGF = '/content/drive/MyDrive/PNGF_outputs'
if os.path.isdir(GDRIVE_PNGF):
    for f in os.listdir(GDRIVE_PNGF):
        if f.startswith('Gmat') and not os.path.isfile(f'/root/PNGF/pngf-opt/{f}'):
            shutil.copy(f'{GDRIVE_PNGF}/{f}', f'/root/PNGF/pngf-opt/{f}')
            print(f'Copied {f} from Drive.')
%cd /root/PNGF/pngf-opt
!./pngf-opt